# Lapa · Предварительный baseline

Chat 5-shot, thinking off; 28 вопросов. Результат: **21/28 (75%)**.
Полный MMLU-ProX и сравнение шести режимов не выполнены

A100 40 GB, Python 3.13. Выполните ячейки по порядку.
Сохраните ZIP и отключите GPU. Старый архив не нужен


## 1. Код и настройки

Все необходимые файлы встроены ниже, выполните весь раздел


In [ ]:
import os
import sys
from pathlib import Path

WORK_ROOT = Path("/content/lapa_mmluprox_v3")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ("eval_support", "lm_eval_tasks/mmlu_prox_ru_chat"):
    (WORK_ROOT / folder).mkdir(parents=True, exist_ok=True)
os.chdir(WORK_ROOT)
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
for module_name in ("eval_support", "eval_support.standalone",
                    "eval_support.mmluprox"):
    module = sys.modules.get(module_name)
    if module and not Path(module.__file__).is_relative_to(WORK_ROOT):
        raise RuntimeError("Перезапустите runtime перед сменой notebook")


In [ ]:
%%writefile requirements-eval.txt
lm-eval @ git+https://github.com/EleutherAI/lm-evaluation-harness.git@d6de81643928d653435c431bae19945d41d32520
transformers==5.3.0
accelerate==1.12.0
bitsandbytes==0.49.2
datasets==4.6.1
huggingface-hub==1.29.0
torch==2.10.0
jedi==0.19.2
cuda-python==12.9.4; sys_platform == "linux"
flash-linear-attention==0.4.2; sys_platform == "linux"
causal-conv1d @ https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp313-cp313-linux_x86_64.whl#sha256=ca4b91cb5df8a561cd046902cb9cfa749f4cef04e9c9d9d870762f0dc5b23f0c ; sys_platform == "linux" and platform_machine == "x86_64" and python_version == "3.13"


In [ ]:
%%writefile eval_support/__init__.py
"""Notebook support; benchmark scoring belongs to lm-eval."""


In [ ]:
%%writefile eval_support/standalone.py
"""Local Colab result export; no Drive or repository checkout required."""

import hashlib
import importlib.metadata
import json
import sys
import traceback
import zipfile
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4


def check_environment(versions, harness_revision):
    """Reject stale imports and the wrong harness commit before loading."""
    for package, expected in versions.items():
        installed = importlib.metadata.version(package).split("+")[0]
        if installed != expected:
            raise RuntimeError(f"{package}: {installed}, expected {expected}")
        module = sys.modules.get(package.replace("-", "_"))
        loaded = getattr(module, "__version__", installed).split("+")[0]
        if loaded != installed:
            raise RuntimeError(
                f"{package} {loaded} is already imported. Restart runtime."
            )
    distribution = importlib.metadata.distribution("lm-eval")
    provenance = json.loads(distribution.read_text("direct_url.json") or "{}")
    commit = provenance.get("vcs_info", {}).get("commit_id")
    if commit != harness_revision:
        raise RuntimeError("lm-eval commit differs from the pinned revision")
    return provenance


def export_results(work_root, results_dir, stage, download=True):
    """Create a new verifiable snapshot, including partial results."""
    work_root = Path(work_root)
    results_dir = Path(results_dir)
    export_dir = work_root / "exports"
    export_dir.mkdir(exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive = export_dir / (
        f"LAPA_MMLUPROX_{stage}_{stamp}_{uuid4().hex[:8]}.zip"
    )
    sources = [work_root / "requirements-eval.txt"]
    sources += [work_root / "protocol.md"]
    sources += [work_root / "source_manifest.json"]
    for folder in ("eval_support", "lm_eval_tasks"):
        sources.extend(
            p for p in (work_root / folder).rglob("*")
            if p.is_file() and p.suffix in {".py", ".yaml"}
        )
    members = {
        "source/" + str(p.relative_to(work_root)): p
        for p in sources if p.is_file()
    }
    members.update({
        "results/" + str(p.relative_to(results_dir)): p
        for p in results_dir.rglob("*") if p.is_file()
    })
    hashes = {}
    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as output:
        for name, path in sorted(members.items()):
            digest = hashlib.sha256()
            with path.open("rb") as source:
                with output.open(name, "w", force_zip64=True) as target:
                    for chunk in iter(lambda: source.read(1024 * 1024), b""):
                        target.write(chunk)
                        digest.update(chunk)
            hashes[name] = digest.hexdigest()
        manifest = {"schema_version": 1, "stage": stage, "files": hashes}
        output.writestr("export_manifest.json", json.dumps(manifest, indent=2))
    with archive.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    checksum = archive.with_suffix(".zip.sha256")
    checksum.write_text(f"{digest}  {archive.name}\n", encoding="utf-8")
    print("Архив:", archive)
    print("SHA-256:", digest)
    if download:
        try:
            from google.colab import files

            files.download(str(archive))
        except Exception as error:
            print("Автоскачивание не удалось:", error)
            print("Скачайте ZIP через панель Files в Colab:", archive)
    return archive


@contextmanager
def diagnostic_stage(work_root, results_dir, stage):
    """Export diagnostics after a Python error, including interruption."""
    try:
        yield
    except BaseException:
        original_traceback = traceback.format_exc()
        try:
            filename = f"{stage}_error_{uuid4().hex[:8]}.json"
            failure = Path(results_dir) / filename
            failure.write_text(json.dumps({
                "stage": stage, "traceback": original_traceback,
            }, ensure_ascii=False, indent=2), encoding="utf-8")
            export_results(work_root, results_dir, stage + "_failed")
        except Exception as error:
            print("Не удалось собрать архив:", error)
            print("Сохраните каталог через Files:", results_dir)
        raise


def require_fast_path(module) -> dict:
    """Check the functions actually selected by Transformers."""
    names = (
        "causal_conv1d_fn",
        "causal_conv1d_update",
        "chunk_gated_delta_rule",
        "fused_recurrent_gated_delta_rule",
        "FusedRMSNormGated",
    )
    if not module.is_fast_path_available or any(
        not callable(getattr(module, name, None)) for name in names
    ):
        raise RuntimeError(
            "Required FLA/causal-conv1d fast path is unavailable. "
            "Reinstall the locked Colab bundle; fallback is not permitted."
        )
    return {name: getattr(module, name).__module__ for name in names}


def check_kernels(torch, implementation) -> dict:
    """Exercise BF16 prefill, cached decode, normalization and NF4 on GPU.

    This small synthetic check needs no model download and is not a timing
    measurement or proof of full-checkpoint correctness.
    """
    import platform

    import bitsandbytes as bnb

    if platform.python_version_tuple()[:2] != ("3", "13"):
        raise RuntimeError("The Colab GPU stack requires Python 3.13")
    if platform.machine() != "x86_64" or torch.version.cuda != "12.8":
        raise RuntimeError("Expected Linux x86_64 and PyTorch CUDA 12.8")
    if not torch.compiled_with_cxx11_abi():
        raise RuntimeError("The causal-conv1d wheel requires CXX11 ABI")
    selected = require_fast_path(implementation)

    def random_tensor(*shape):
        return torch.randn(*shape, device="cuda:0", dtype=torch.bfloat16)

    with torch.inference_mode():
        x = random_tensor(1, 32, 16)
        weight = random_tensor(32, 4)
        actual = implementation.causal_conv1d_fn(
            x, weight, activation="silu"
        )
        expected = torch.nn.functional.silu(
            torch.nn.functional.conv1d(
                x, weight.unsqueeze(1), padding=3, groups=32
            )[..., :16]
        )
        torch.testing.assert_close(actual, expected, rtol=0.03, atol=0.03)
        state = x[..., -4:].contiguous()
        reference_state = state.clone()
        token = random_tensor(1, 32, 1)
        actual = implementation.causal_conv1d_update(
            token, state, weight, activation="silu"
        )
        expected = implementation.torch_causal_conv1d_update(
            token, reference_state, weight, activation="silu"
        )
        torch.testing.assert_close(actual, expected, rtol=0.03, atol=0.03)
        torch.testing.assert_close(state, reference_state)

        q, k, v = [random_tensor(1, 16, 2, 128) for _ in range(3)]
        options = {
            "g": -random_tensor(1, 16, 2).float().abs(),
            "beta": random_tensor(1, 16, 2).sigmoid(),
            "initial_state": None,
            "output_final_state": True,
            "use_qk_l2norm_in_kernel": True,
        }
        actual, state = implementation.chunk_gated_delta_rule(
            q, k, v, **options
        )
        expected, reference_state = (
            implementation.torch_recurrent_gated_delta_rule(
                q, k, v, **options
            )
        )
        torch.testing.assert_close(actual, expected, rtol=0.03, atol=0.03)
        torch.testing.assert_close(
            state, reference_state, rtol=0.03, atol=0.03
        )
        options.update(
            g=options["g"][:, :1].contiguous(),
            beta=options["beta"][:, :1].contiguous(),
            initial_state=state,
        )
        actual, _ = implementation.fused_recurrent_gated_delta_rule(
            q[:, :1], k[:, :1], v[:, :1], **options
        )
        options["initial_state"] = reference_state
        expected, _ = implementation.torch_recurrent_gated_delta_rule(
            q[:, :1], k[:, :1], v[:, :1], **options
        )
        torch.testing.assert_close(actual, expected, rtol=0.03, atol=0.03)
        norm = implementation.FusedRMSNormGated(
            128, activation="silu", device="cuda:0", dtype=torch.bfloat16
        )
        assert_finite(torch, norm(q, q))
        linear = bnb.nn.Linear4bit(
            128, 128, bias=False, compute_dtype=torch.bfloat16,
            compress_statistics=True, quant_type="nf4",
        ).to("cuda:0")
        assert_finite(torch, linear(q))
    torch.cuda.synchronize()
    return {"status": "passed", "selected_functions": selected}


def assert_finite(torch, tensor) -> None:
    """Reject a successfully imported kernel that produces invalid values."""
    if not torch.isfinite(tensor).all().item():
        raise RuntimeError("GPU kernel check produced non-finite values")


In [ ]:
%%writefile eval_support/mmluprox.py
"""Reproducible harness orchestration, without a custom scorer."""


import hashlib
import json
import time
import traceback
from pathlib import Path

import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.utils import handle_non_serializable


DATA_REVISION = "8e6106a6c6ce1c5027e66cc338143cf997b2aa09"


MODEL_REVISION = "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0"


HARNESS_REVISION = "d6de81643928d653435c431bae19945d41d32520"


def save_json(path, value):
    """Never overwrite a previous result."""
    with Path(path).open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2,
                  default=handle_non_serializable)


def seal(directory):
    """Hash saved files, including partial raw output after failure."""
    directory = Path(directory)
    files = {}
    for path in sorted(directory.rglob("*")):
        if path.is_file() and path.name != "manifest.json":
            with path.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            files[str(path.relative_to(directory))] = digest
    save_json(directory / "manifest.json", files)


class AuditedHFLM(HFLM):
    """Use harness generation unchanged; retain its raw token outputs."""

    raw_path = None
    deadline = None

    def tok_batch_encode(self, strings, padding_side="left",
                         left_truncate_len=None, truncation=False):
        inputs, mask = super().tok_batch_encode(
            strings, padding_side=padding_side,
            left_truncate_len=None, truncation=False,
        )
        if left_truncate_len and inputs.shape[1] > left_truncate_len:
            raise RuntimeError("Input exceeds context budget; no truncation")
        return inputs, mask

    def _model_generate(self, context, **kwargs):
        import torch

        if self.deadline is not None:
            remaining = self.deadline - time.monotonic()
            if remaining <= 0:
                raise TimeoutError("Session time budget exhausted")
            kwargs["max_time"] = remaining
        on_gpu = context.device.type == 'cuda'
        if on_gpu:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        started = time.monotonic()
        output = super()._model_generate(context=context, **kwargs)
        if on_gpu:
            torch.cuda.synchronize()
        elapsed = time.monotonic() - started
        for inputs, tokens in zip(
            context.tolist(), output.tolist(), strict=True,
        ):
            generated = tokens[len(inputs):]
            raw = self.tokenizer.decode(
                generated, skip_special_tokens=False,
            )
            if self.raw_path is not None:
                with self.raw_path.open("a", encoding="utf-8") as stream:
                    record = {
                        "input_ids": inputs, "generated_ids": generated,
                        "raw_text": raw, "missing_think_end": False,
                        "generation_seconds": elapsed,
                        "peak_allocated_bytes": (
                            torch.cuda.max_memory_allocated() if on_gpu
                            else None
                        ),
                    }
                    stream.write(json.dumps(record, ensure_ascii=False))
                    stream.write("\n")
        if self.deadline is not None and time.monotonic() >= self.deadline:
            raise TimeoutError("Generation stopped at session time budget")
        return output


def run_eval(lm, task_manager, directory, name, task, sample_index,
             document_hash):
    """Evaluate chat 5-shot without native thinking."""
    run_dir = Path(directory) / name
    run_dir.mkdir(parents=True, exist_ok=False)
    lm.enable_thinking = False
    lm.chat_template_args["enable_thinking"] = False
    lm.think_end_token = None
    lm.raw_path = run_dir / "raw_generations.jsonl"
    started = time.monotonic()
    save_json(run_dir / "settings.json", {
        "task": task, "shots": 5, "chat": True,
        "thinking": False, "limit": None,
        "dataset_revision": DATA_REVISION,
        "model_revision": MODEL_REVISION,
        "harness_revision": HARNESS_REVISION,
        "sample_index": sample_index, "document_hash": document_hash,
        "debug": False, "split": "test",
    })
    try:
        # Pinned harness factory propagates revision before dataset loading.
        # A bare {task: group_name} is an inline task, not a group reference.
        overrides = {"dataset_kwargs": {"revision": DATA_REVISION}}
        prepared_task = task_manager._factory.build(
            task_manager.task_index[task],
            overrides=overrides,
            registry=task_manager.task_index,
        )
        document = prepared_task.test_docs()[sample_index]
        if digest_json(document) != document_hash:
            raise ValueError("Selected document differs from the plan")
        result = lm_eval.simple_evaluate(
            model=lm,
            tasks=[prepared_task],
            task_manager=task_manager,
            num_fewshot=5,
            apply_chat_template=True,
            fewshot_as_multiturn=True,
            samples={task: [sample_index]},
            log_samples=True,
            bootstrap_iters=0,
            random_seed=0,
            numpy_random_seed=1234,
            torch_random_seed=1234,
            fewshot_random_seed=1234,
        )
        save_json(run_dir / f"{name}.json", result)
        save_json(run_dir / "timing.json", {
            "wall_seconds": time.monotonic() - started,
            "note": "Evaluation wall time, not paired inference E2E",
        })
        seal(run_dir)
        save_json(run_dir / "completed.json", {"status": "completed"})
        return result
    except BaseException:
        original_traceback = traceback.format_exc()
        try:
            save_json(run_dir / "failure.json", {
                "traceback": original_traceback,
                "wall_seconds": time.monotonic() - started,
            })
        except OSError as error:
            print("Could not save failure diagnostics:", error)
        raise
    finally:
        lm.raw_path = None


SUBJECTS = tuple(sorted(
    'biology business chemistry computer_science economics engineering '
    'health history law math other philosophy physics psychology'.split()
))


TOTAL_SECONDS = 6 * 3600


def digest_json(value):
    """Stable identity for documents and frozen plans."""
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False,
                         separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def verify_run(directory):
    """Accept only a sealed, completed harness result."""
    directory = Path(directory)
    if not (directory / 'completed.json').exists():
        return False
    manifest = json.loads((directory / 'manifest.json').read_text())
    for name, expected in manifest.items():
        path = directory / name
        if not path.resolve().is_relative_to(directory.resolve()):
            raise ValueError('Unsafe run manifest path')
        with path.open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != expected:
            raise ValueError('Run checksum mismatch')
    required = {'settings.json', 'timing.json', 'raw_generations.jsonl',
                directory.name + '.json'}
    if not required <= manifest.keys():
        raise ValueError('Incomplete run manifest')
    return True


def freeze_json(path, value):
    """Keep a plan immutable across sessions."""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError('Frozen plan differs; do not mix experiments')
    else:
        save_json(path, value)
    return value


def make_selection(dataset, source_hashes):
    """Rank rows without labels; reserve calibration before evaluation."""
    calibration, candidates = [], {}
    for subject in SUBJECTS:
        rows = [row for row in dataset
                if row['category'].replace(' ', '_') == subject]
        ranked = []
        for index, row in enumerate(rows):
            # Previously viewed business smoke is excluded from both sets.
            if subject == 'business' and row['question_id'] in (70, 71):
                continue
            identity = {'subject': subject,
                        'question_id': row['question_id']}
            ranked.append((digest_json({'seed': 20260926, **identity}), {
                **identity, 'index': index, 'sha256': digest_json(row),
            }))
        ranked.sort(key=lambda item: item[0])
        if len(ranked) < 3:
            raise ValueError('Too few documents for disjoint subsets')
        calibration.append(ranked[0][1])
        candidates[subject] = [item[1] for item in ranked[1:]]
    return {
        'schema': 1, 'seed': 20260926,
        'dataset_revision': DATA_REVISION,
        'model_revision': MODEL_REVISION,
        'harness_revision': HARNESS_REVISION,
        'sources': source_hashes, 'budget_seconds': TOTAL_SECONDS,
        'calibration': calibration, 'candidates': candidates,
        'note': 'Stratified preliminary comparison; not a full benchmark',
    }


def finished_attempt(study, phase, mode, item):
    """Find at most one completed attempt for the exact frozen document."""
    root = Path(study) / phase / mode / (
        item['subject'] + '_' + str(item['index'])
    )
    complete = [p for p in sorted(root.glob('attempt_*')) if verify_run(p)]
    if len(complete) > 1:
        raise ValueError('Duplicate completed document')
    if not complete:
        return None
    result = complete[0]
    settings = json.loads((result / 'settings.json').read_text())
    if settings['document_hash'] != item['sha256']:
        raise ValueError('Completed result belongs to another document')
    return result


def make_baseline_selection(dataset, source_hashes, per_subject=2,
                            session_budget_seconds=TOTAL_SECONDS):
    """Fix an equal subject sample, excluding prior development questions."""
    if per_subject < 1:
        raise ValueError('Positive sample size required')
    if session_budget_seconds <= 300:
        raise ValueError('Session budget must exceed the export reserve')
    previous = make_selection(dataset, source_hashes)
    items = []
    for subject in SUBJECTS:
        candidates = previous['candidates'][subject]
        if len(candidates) < per_subject:
            raise ValueError('Insufficient subject questions')
        items.extend(candidates[:per_subject])
    return {
        'protocol': 'chat-5shot-baseline-v1',
        'dataset_revision': DATA_REVISION,
        'model_revision': MODEL_REVISION,
        'harness_revision': HARNESS_REVISION,
        'sources': source_hashes, 'seed': previous['seed'],
        'per_subject': per_subject, 'items': items,
        'excluded_development': previous['calibration'],
        'shots': 5, 'chat': True, 'thinking': False,
        'max_gen_toks': 2048, 'batch_size': 1,
        'session_budget_seconds': session_budget_seconds,
        'question_budget_seconds': 600,
        'note': 'Preliminary stratified baseline, not a full benchmark',
    }


def run_baseline(lm, manager, directory, selection, session_started):
    """Evaluate fixed questions through harness and retain each completion."""
    from uuid import uuid4

    directory = Path(directory)
    mode = 'chat_5shot_direct'
    scores, invalid, token_caps = [], 0, 0
    for number, item in enumerate(selection['items'], 1):
        completed = finished_attempt(directory, 'baseline', mode, item)
        if completed is None:
            remaining = selection['session_budget_seconds'] - 300
            remaining -= time.monotonic() - session_started
            if remaining <= 0:
                raise TimeoutError('Session budget exhausted; partial ZIP')
            print(f"{number}/{len(selection['items'])}: {item['subject']}",
                  flush=True)
            root = directory / 'baseline' / mode / (
                item['subject'] + '_' + str(item['index'])
            )
            name = 'attempt_' + uuid4().hex[:12]
            lm.deadline = time.monotonic() + min(600, remaining)
            try:
                run_eval(
                    lm, manager, root, name,
                    'mmlu_prox_ru_chat_' + item['subject'],
                    sample_index=item['index'], document_hash=item['sha256'],
                )
            finally:
                lm.deadline = None
            completed = root / name
        result = json.loads((completed / (completed.name + '.json'))
                            .read_text())
        records = next(iter(result['samples'].values()))
        if len(records) != 1:
            raise ValueError('Expected one scored question')
        scores.append(records[0]['exact_match'])
        invalid += records[0]['filtered_resps'] == ['[invalid]']
        raw = (completed / 'raw_generations.jsonl').read_text()
        for line in raw.splitlines():
            token_caps += len(json.loads(line)['generated_ids']) >= 2048
    summary = {
        'protocol': selection['protocol'], 'status': 'completed',
        'questions': len(scores), 'correct': sum(scores),
        'accuracy': sum(scores) / len(scores),
        'invalid_answers': invalid, 'outputs_at_token_limit': token_caps,
        'note': 'Mean of saved harness exact_match; not full MMLU-ProX',
    }
    freeze_json(directory / 'summary.json', summary)
    return summary


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml
group: mmlu_prox_ru_chat
task:
  - mmlu_prox_ru_chat_biology
  - mmlu_prox_ru_chat_business
  - mmlu_prox_ru_chat_chemistry
  - mmlu_prox_ru_chat_computer_science
  - mmlu_prox_ru_chat_economics
  - mmlu_prox_ru_chat_engineering
  - mmlu_prox_ru_chat_health
  - mmlu_prox_ru_chat_history
  - mmlu_prox_ru_chat_law
  - mmlu_prox_ru_chat_math
  - mmlu_prox_ru_chat_other
  - mmlu_prox_ru_chat_philosophy
  - mmlu_prox_ru_chat_physics
  - mmlu_prox_ru_chat_psychology
aggregate_metric_list:
  - aggregation: mean
    metric: exact_match
    weight_by_size: true
    filter_list: custom-extract
metadata:
  version: 0.0


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml
dataset_path: li-lab/MMLU-ProX
dataset_name: ru
dataset_kwargs:
  revision: 8e6106a6c6ce1c5027e66cc338143cf997b2aa09

test_split: test
fewshot_split: validation

description: |
  Перед вами вопрос с множественным выбором.
  Выберите правильный вариант ответа.
  Финальный ответ должен заканчиваться в формате "Ответ - (X)",
  где X — латинская буква правильного варианта.

fewshot_config:
  sampler: first_n
  doc_to_text: !function utils.doc_to_text
  doc_to_target: !function utils.fewshot_target

output_type: generate_until

doc_to_text: !function utils.doc_to_text
doc_to_target: answer

filter_list:
  - name: custom-extract
    filter:
      - function: regex
        regex_pattern: 'Ответ - \(?([ABCDEFGHIJ])\)?'
      - function: take_first

generation_kwargs:
  until:
    - "</s>"
    - "Q:"
    - "Вопрос:"
    - "<|im_end|>"
  do_sample: false
  temperature: 0.0
  max_gen_toks: 2048

num_fewshot: 5

metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_case: true
    ignore_punctuation: true

metadata:
  version: 0.0


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_biology
task_alias: biology
process_docs: !function utils.process_biology


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_business
task_alias: business
process_docs: !function utils.process_business


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_chemistry
task_alias: chemistry
process_docs: !function utils.process_chemistry


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_computer_science
task_alias: computer_science
process_docs: !function utils.process_computer_science


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_economics
task_alias: economics
process_docs: !function utils.process_economics


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_engineering
task_alias: engineering
process_docs: !function utils.process_engineering


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_health
task_alias: health
process_docs: !function utils.process_health


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_history
task_alias: history
process_docs: !function utils.process_history


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_law
task_alias: law
process_docs: !function utils.process_law


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_math
task_alias: math
process_docs: !function utils.process_math


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_other
task_alias: other
process_docs: !function utils.process_other


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_philosophy
task_alias: philosophy
process_docs: !function utils.process_philosophy


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_physics
task_alias: physics
process_docs: !function utils.process_physics


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml
include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_psychology
task_alias: psychology
process_docs: !function utils.process_psychology


In [ ]:
%%writefile lm_eval_tasks/mmlu_prox_ru_chat/utils.py
from functools import partial

CHOICES = list("ABCDEFGHIJ")
MAX_OPTIONS = 10


def doc_to_text(example):
    prompt = "Вопрос:\n"
    prompt += example["question"] + "\n"
    prompt += "Варианты:\n"

    for i in range(MAX_OPTIONS):
        option = example[f"option_{i}"]
        if option is not None:
            prompt += f"{CHOICES[i]}. {option}\n"

    return prompt


def fewshot_target(example):
    return f"Ответ - ({example['answer']})"


def process_docs(dataset, subject):
    return dataset.filter(lambda x: x["category"] == subject)


process_biology = partial(process_docs, subject="biology")
process_business = partial(process_docs, subject="business")
process_chemistry = partial(process_docs, subject="chemistry")
process_computer_science = partial(
    process_docs, subject="computer science"
)
process_economics = partial(process_docs, subject="economics")
process_engineering = partial(process_docs, subject="engineering")
process_health = partial(process_docs, subject="health")
process_history = partial(process_docs, subject="history")
process_law = partial(process_docs, subject="law")
process_math = partial(process_docs, subject="math")
process_other = partial(process_docs, subject="other")
process_philosophy = partial(process_docs, subject="philosophy")
process_physics = partial(process_docs, subject="physics")
process_psychology = partial(process_docs, subject="psychology")


In [ ]:
%%writefile protocol.md
# Предварительный baseline MMLU-ProX RU

Chat 5-shot, thinking off; Qwen NF4/BF16; стандартный lm-eval scoring.
28 вопросов: по два на предмет. SHA-выбор по ID, seed 20260926.
Исключены business 70/71 и ранее выделенные 14 development-вопросов.
Validation first_n: пять пар user/assistant своего предмета, без CoT.
Greedy, max_gen_toks=2048, batch=1, context32768, без обрезки входа.
Версии, выборка, настройки, ответы и метрики сохраняются в ZIP.

Бюджет задаёт SESSION_BUDGET_HOURS; включает подготовку и ожидание.
Резерв экспорта — 300 с, предел ответа — 600 с. Таймер кооперативный.
После скачивания ZIP отключите GPU. Потеря runtime до экспорта
может потерять прогресс; в той же сессии готовые ответы пропускаются.

Результат 30.09: 21/28 (75%), один ответ без буквы на лимите 2048.
Это малая предварительная выборка, не полный benchmark и не speedup.
Полная оценка, шесть режимов и эффект адаптации не выполнены.
Для будущего сравнения сохранить вопросы и настройки; не обучаться
и не подбирать метод по этим test-ответам. Время сравнивать парно.
Исходный результат и настройки прогона сохранены в неизменном ZIP.


In [ ]:
%%writefile source_manifest.json
{
  "requirements-eval.txt": "66c2e8fd224964045b54e1617a290871d99a2295805386942c9645a5d0e6ae8b",
  "eval_support/__init__.py": "78d7a894f2ba87185a30a183c3696b907b0b0e04f8ccfda7baca3c23c258d849",
  "eval_support/standalone.py": "1eae68c7306ca8876b8778451943de8fe495b41483c902f80fd338e6c7c364a5",
  "eval_support/mmluprox.py": "87e7f3721b3ae07b4779f70f6b9e70b1f10c564de635108b0feeef3b31f3e791",
  "lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml": "852bf00302ac68b796a07121646afeb1c054f165b1b7e1585c89a7a549193669",
  "lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml": "ba71b82a8872d81e841ca566cf1228fbf6bac288963c40669112c4d241156093",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml": "5c5652af779b6f76263c8b2c50d1e556b49ed9ae135a569de7a729f819d2db23",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml": "c3ee2987b8742f81fa4eb59262d3bd247a65efa6a9cb2a75ed7d75427bf08d75",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml": "5be779e17271bb81d0ebe8702a24c3ac399287d0df4836ab81524ea6a098257a",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml": "1816af75a73635c7705c35aef3072f15a22ee54aa305819880927dcf7e3a9ae5",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml": "ef1fcf83eadc67e89bb3b4381bad0eac16585c3bfc266bf5d124f99d4030d487",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml": "fe1bcfa0009c42c7998c37ede092bcf3bb9b896dbbc5885980f92a89ea8f05b5",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml": "80e6d0e0613752a2454df265c58f34f9448c4791e347893abda72305e6fbc64a",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml": "047c600f10c0b38e0b3eff2c428079793c3d89ecd2bcc1b43019df6c83f84956",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml": "7679544fafe2ada9c1e667e4adc13db1f319f420ecb97908ef48198737ba2289",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml": "af96df4b28b32daa1846bc0f29264709ebe2e7a7c4789f137fa900ae61d49694",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml": "87a294c66ccf38044a6016ad0ae0c5cb5a4c3f092ebd2c40dbdc6c4de3ff1580",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml": "fb0599414d22276942170f10fc4238a590e4981d04be08ceae9202af54fe0fe2",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml": "bfb64e6779bef91b5092dea22efee61f8ebf1e7888861558bd308a3a6d4e7196",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml": "cb4802cc2f6f07283d8a433cd55512d436a34d653d6ee5d09f1e0cf561e6a751",
  "lm_eval_tasks/mmlu_prox_ru_chat/utils.py": "99d10756ca88eee8fa3ff9fce3a4d4efb746b17a7a9b9feb7fcfbb76b15e5f3a",
  "protocol.md": "b3cab5224c1cb8b4f1bd20aa9b8ffcd9c1ade0c2b6c90987ec965e81713dd0b0"
}


## 2. Установить зависимости

Версии закреплены, CUDA wheel устанавливается без сборки


In [ ]:
import hashlib
import importlib.metadata
import json
import subprocess
import time
from datetime import datetime, timezone
from uuid import uuid4

if "WORK_ROOT" not in globals():
    raise RuntimeError("Сначала выполните все ячейки раздела 1")

manifest_path = WORK_ROOT / "source_manifest.json"
if not manifest_path.is_file():
    raise RuntimeError("Выполните все ячейки раздела 1: нет manifest")
expected = json.loads(manifest_path.read_text())
missing = [name for name in expected if not (WORK_ROOT / name).is_file()]
if missing:
    raise RuntimeError(
        "Не созданы файлы: " + ", ".join(missing)
        + ". Выполните все ячейки раздела 1, затем повторите эту ячейку."
    )
for name, digest in expected.items():
    actual = hashlib.sha256((WORK_ROOT / name).read_bytes()).hexdigest()
    if actual != digest:
        raise RuntimeError(
            "Файл отличается от notebook: " + name
            + ". Повторите раздел 1 для восстановления встроенного кода."
        )

from eval_support.standalone import (  # noqa: E402
    check_environment,
    diagnostic_stage,
    export_results,
)

if "RESULTS_DIR" not in globals():
    SESSION_STARTED = time.monotonic()
    SESSION = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    SESSION += "-" + uuid4().hex[:8]
    RESULTS_DIR = WORK_ROOT / "results" / SESSION
    RESULTS_DIR.mkdir(parents=True, exist_ok=False)
    (RESULTS_DIR / "session.json").write_text(json.dumps({
        "session": SESSION,
        "release": "2026-09-24-standalone-v3",
        "source_sha256": expected,
        "delivery": "local notebook upload; no project Git checkout",
    }, ensure_ascii=False, indent=2), encoding="utf-8")
print("Результаты сессии:", RESULTS_DIR)

with diagnostic_stage(WORK_ROOT, RESULTS_DIR, "install"):
    if sys.version_info[:2] != (3, 13):
        raise RuntimeError("Нужен Python 3.13 для закреплённого CUDA wheel")
    setup_id = uuid4().hex[:8]
    install_log = RESULTS_DIR / f"install_{setup_id}.log"
    if any(name in sys.modules for name in ("torch", "transformers")):
        raise RuntimeError("Перед установкой перезапустите runtime")
    for label, arguments in {
        "packages_before": ["freeze", "--all"],
        "pip_check_before": ["check"],
    }.items():
        snapshot = subprocess.run(
            [sys.executable, "-m", "pip", *arguments],
            text=True, capture_output=True,
        )
        (RESULTS_DIR / f"{label}_{setup_id}.txt").write_text(
            snapshot.stdout + snapshot.stderr, encoding="utf-8",
        )
    print("Устанавливаем зависимости; это может занять несколько минут.")
    compatibility = []
    for package, version in {
        "torchvision": "0.25.0", "torchaudio": "2.10.0",
    }.items():
        try:
            importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            continue
        compatibility.append(f"{package}=={version}")
    with install_log.open("w", encoding="utf-8") as stream:
        installed = subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "--report", str(RESULTS_DIR / f"install_{setup_id}.json"), "-r",
             str(WORK_ROOT / "requirements-eval.txt"), *compatibility],
            stdout=stream, stderr=subprocess.STDOUT,
        )
    print(install_log.read_text()[-6000:])
    installed.check_returncode()
    snapshot = subprocess.run(
        [sys.executable, "-m", "pip", "freeze", "--all"],
        text=True, capture_output=True, check=True,
    )
    (RESULTS_DIR / f"packages_after_{setup_id}.txt").write_text(
        snapshot.stdout, encoding="utf-8",
    )
    checked = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True, capture_output=True,
    )
    (RESULTS_DIR / ("pip_check_" + uuid4().hex[:8] + ".log")).write_text(
        checked.stdout + checked.stderr, encoding="utf-8",
    )
    print(checked.stdout, checked.stderr)
    checked.check_returncode()


Если требуется перезапуск Python runtime, перезапустите его и выполните
разделы 1–3 заново. Не продолжайте со старыми импортированными библиотеками


## 3. Проверить среду


In [ ]:
with diagnostic_stage(WORK_ROOT, RESULTS_DIR, "environment"):
    versions = {
        "transformers": "5.3.0", "accelerate": "1.12.0",
        "bitsandbytes": "0.49.2", "datasets": "4.6.1",
        "huggingface-hub": "1.29.0", "torch": "2.10.0",
        "jedi": "0.19.2", "cuda-python": "12.9.4",
        "flash-linear-attention": "0.4.2", "causal-conv1d": "1.7.0",
    }
    provenance = check_environment(
        versions, "d6de81643928d653435c431bae19945d41d32520",
    )
    from lm_eval.tasks import TaskManager

    from eval_support.mmluprox import (
        MODEL_REVISION,
        AuditedHFLM,
        save_json,
    )

    task_manager = TaskManager(
        include_path=str(WORK_ROOT / "lm_eval_tasks"),
    )
    assert "mmlu_prox_ru" in task_manager.all_tasks
    assert "mmlu_prox_ru_chat" in task_manager.all_tasks
    assert len([t for t in task_manager.all_tasks
                if t.startswith("mmlu_prox_ru_chat_")]) == 14
    save_json(RESULTS_DIR / ("environment_" + uuid4().hex[:8] + ".json"), {
        "python": sys.version,
        "harness_provenance": provenance,
        "packages": subprocess.check_output(
            [sys.executable, "-m", "pip", "freeze"], text=True),
        "source_sha256": expected,
    })
print("Harness и 14 chat-задач доступны.")


## 4. Выборка и бюджет

Бюджет включает подготовку. Пять validation-примеров — демонстрации


In [ ]:
from datasets import load_dataset

from eval_support.mmluprox import (
    DATA_REVISION,
    freeze_json,
    make_baseline_selection,
    run_baseline,
)

PER_SUBJECT = 2
SESSION_BUDGET_HOURS = 6
with diagnostic_stage(WORK_ROOT, RESULTS_DIR, "selection"):
    dataset = load_dataset(
        "li-lab/MMLU-ProX", "ru", revision=DATA_REVISION, split="test",
    )
    selection = freeze_json(
        RESULTS_DIR / "selection.json",
        make_baseline_selection(
            dataset, expected, PER_SUBJECT,
            session_budget_seconds=SESSION_BUDGET_HOURS * 3600,
        ),
    )
print("Зафиксировано вопросов:", len(selection["items"]))


## 5. Загрузить модель

NF4/BF16, ускоренные kernels, одна загрузка на сессию.


In [ ]:
import os

# Materialize weights sequentially before NF4 conversion.
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

with diagnostic_stage(WORK_ROOT, RESULTS_DIR, "model_load"):
    elapsed = time.monotonic() - SESSION_STARTED
    if elapsed >= selection["session_budget_seconds"] - 300:
        raise TimeoutError("Бюджет исчерпан до загрузки модели")
    import importlib

    import torch
    from bitsandbytes.nn import Linear4bit
    from transformers import (
        AutoConfig,
        AutoModelForCausalLM,
        AutoTokenizer,
        BitsAndBytesConfig,
    )

    assert torch.cuda.is_available(), "Нужен GPU runtime"
    assert "A100" in torch.cuda.get_device_name(0), "Протокол: A100"
    assert torch.cuda.is_bf16_supported()
    total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if not 35 <= total_gib <= 45:
        raise RuntimeError("Протокол рассчитан на A100 40 GB")
    from eval_support.standalone import check_kernels

    MODEL_ID = "Qwen/Qwen3.8-27B"
    if "kernel_report" not in globals():
        config = AutoConfig.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION,
        )
        model_class = AutoModelForCausalLM._model_mapping[type(config)]
        implementation = importlib.import_module(model_class.__module__)
        kernel_report = check_kernels(torch, implementation)
        save_json(RESULTS_DIR / "kernels.json", kernel_report)
        torch.cuda.empty_cache()
    if "hf_model" not in globals():
        if globals().get("model_load_started", False):
            raise RuntimeError("После сбоя загрузки перезапустите среду")
        model_load_started = True
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        save_json(RESULTS_DIR / "model_load_start.json", {
            "hf_deactivate_async_load": os.environ[
                "HF_DEACTIVATE_ASYNC_LOAD"],
            "free_bytes": free_bytes,
            "total_bytes": total_bytes,
            "allocated_bytes": torch.cuda.memory_allocated(),
            "reserved_bytes": torch.cuda.memory_reserved(),
        })
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION,
        )
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        hf_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION,
            device_map={"": 0}, dtype=torch.bfloat16,
            quantization_config=quantization_config,
            attn_implementation="sdpa",
        )
        hf_model.eval()
    assert hf_model.is_loaded_in_4bit
    assert any(isinstance(m, Linear4bit) for m in hf_model.modules())
    if "lm" not in globals():
        lm = AuditedHFLM(
            pretrained=hf_model, tokenizer=tokenizer,
            batch_size=1, max_length=32768, enable_thinking=False,
        )
    if not (RESULTS_DIR / "gpu.json").exists():
        save_json(RESULTS_DIR / "gpu.json", {
            "name": torch.cuda.get_device_name(0),
            "memory_total": torch.cuda.get_device_properties(0).total_memory,
            "cuda": torch.version.cuda,
            "model_footprint": hf_model.get_memory_footprint(),
            "memory_allocated": torch.cuda.memory_allocated(),
            "memory_reserved": torch.cuda.memory_reserved(),
            "quantization": quantization_config.to_dict(),
        })
    print(torch.cuda.get_device_name(0), hf_model.get_memory_footprint())


## 6. Оценить

Один запуск. ZIP скачивается в конце или при ошибке.


In [ ]:
with diagnostic_stage(WORK_ROOT, RESULTS_DIR, "baseline"):
    try:
        summary = run_baseline(
            lm, task_manager, RESULTS_DIR, selection, SESSION_STARTED,
        )
        print(json.dumps(summary, ensure_ascii=False, indent=2))
    finally:
        save_json(RESULTS_DIR / ("usage_" + uuid4().hex[:8] + ".json"), {
            "session": SESSION,
            "seconds": time.monotonic() - SESSION_STARTED,
            "session_budget_seconds": selection["session_budget_seconds"],
        })
archive = export_results(WORK_ROOT, RESULTS_DIR, "baseline")


## 7. Повторно скачать ZIP при необходимости


In [ ]:
archive = export_results(WORK_ROOT, RESULTS_DIR, "manual")
